In [ ]:
# hide
import numpy as np
import pyquist as pq
from pyquist.helper import amplitude_to_db, db_to_amplitude

In [ ]:
# collapse
def feedback_delay(x: np.ndarray, M: int, g: float = 1.0) -> np.ndarray:
    """A feedback delay line with attenuation, y[n] = x[n] + g * y[n - M]."""
    buffer = np.zeros(M)              # the delay line, as a circular buffer
    i = 0
    y = np.zeros(len(x))
    for n in range(len(x)):
        y[n] = x[n] + g * buffer[i]   # attenuate y[n - M] on every trip
        buffer[i] = y[n]
        i = (i + 1) % M
    return y

def allpass(x: np.ndarray, M: int, g: float) -> np.ndarray:
    """An all-pass filter, y[n] = -g * x[n] + x[n - M] + g * y[n - M],
    built from a feedback delay line."""
    v = feedback_delay(x, M, g)                          # v[n] = x[n] + g * v[n - M]
    v_delayed = np.concatenate([np.zeros(M), v[:-M]])    # v[n - M]
    return -g * v + v_delayed

def wet_ratio(db: float) -> float:
    """The wet ratio w that mixes the wet signal `db` decibels above the dry signal."""
    return 1 / (1 + db_to_amplitude(-db))

In [ ]:
def schroeder_reverb(x: np.ndarray, t60: float, f_s: int = 44100) -> np.ndarray:
    """Schroeder's reverberator: four feedback combs in parallel, then two
    all-pass filters in series."""
    wet = np.zeros(len(x))
    for tau in [0.0297, 0.0371, 0.0411, 0.0437]:         # comb delays, in seconds
        M = round(tau * f_s)
        g = 10 ** (-3 * M / (f_s * t60))                   # decay by 60 dB in t60 seconds
        wet += (feedback_delay(x, M, g) - x) / 4           # keep only the reflections
    for tau in [0.005, 0.0017]:                           # all-pass delays, in seconds
        wet = allpass(wet, round(tau * f_s), g=0.7)
    return wet


f_s = 44100
marimba = pq.Audio.from_file("../assets/marimba.wav").samples[:, 0]
marimba = np.concatenate([marimba, np.zeros(2 * f_s)])   # room for the reverb's tail
wet = schroeder_reverb(marimba, t60=2.0)
w = wet_ratio(0.0)
pq.play(pq.Audio(marimba, f_s), normalize=True)
pq.play(pq.Audio((1 - w) * marimba + w * wet, f_s), normalize=True)